<div text-align="center">
  <img src="https://raw.githubusercontent.com/FarnoushRJ/MambaLRP/main/assets/MambaLRP_logo.jpeg" width="1000"/>
</div>


<div text-align="center"><h1>🐍 MambaLRP is here! 🎉</h1>

Clone the repository and install MambaLRP.

In [1]:
# Cell 1: Clone
!git clone https://github.com/AdamBosch/MambaLRP.git || echo "Already cloned"

fatal: destination path 'MambaLRP' already exists and is not an empty directory.
Already cloned


In [2]:
# Cell 5: Install MambaLRP
!pip install ./MambaLRP --no-deps -q


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import transformers
print(transformers.__version__)  # should be 4.x

import torch
print(torch.__version__)         # 2.1.1+cu118

import causal_conv1d
import mamba_ssm
from transformers import MambaConfig, MambaForCausalLM, AutoTokenizer
print("All imports OK")

4.57.6
2.2.2+cu118
All imports OK


Import necessary packages.

In [4]:
import sys

from mamba_lrp.model.mamba_huggingface import ModifiedMambaForCausalLM
from mamba_lrp.model.utils import *
from mamba_lrp.lrp.utils import relevance_propagation
from mamba_lrp.dataset.general_dataset import get_sst_dataset, get_medbios_dataset
import torch
import numpy as np

## Load model

Load model and tokenizer.

In [5]:
import torch
import gc

# 1. Delete the variables holding the references
# Note: Ensure you are actually deleting the correct names
if 'R' in locals(): del R
if 'attr' in locals(): del attr
if 'embeddings' in locals(): del embeddings
if 'morf_probs' in locals(): del morf_probs

# 2. Force Python to clear unused references from RAM
gc.collect()

# 3. Force PyTorch to release its internal cache back to the OS
torch.cuda.empty_cache()

# 4. Optional: Check how much is actually used vs reserved
print(f"Actually Used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved (Cached): {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Actually Used: 0.00 GB
Reserved (Cached): 0.00 GB


In [6]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
tokenizer.pad_token = tokenizer.eos_token

model = MambaForCausalLM.from_pretrained(
    "state-spaces/mamba-1.4b-hf",
    use_cache=True
)
hidden_size = model.config.hidden_size
model.lm_head = torch.nn.Linear(hidden_size, 2, bias=True)

model.config.pad_token_id = tokenizer.pad_token_id

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from tqdm import tqdm


def train_sst_local(model, tokenizer, device):
    train_dataset = get_sst_dataset(tokenizer, max_length=None, truncation=True, split='train')
    val_dataset = get_sst_dataset(tokenizer, max_length=None, truncation=True, split='val')
    test_dataset = get_sst_dataset(tokenizer, max_length=None, truncation=True, split='test')

    print(f"Train size: {len(train_dataset)}")
    print(f"Validation size: {len(val_dataset)}")
    print(f"Test size: {len(test_dataset)}")

    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False
    )

    # Replace classification head
    model.lm_head = torch.nn.Linear(hidden_size, 2, bias=True).to(device)
    
    for param in model.backbone.parameters():
        param.requires_grad = False

    optimizer = AdamW(model.lm_head.parameters(), lr=7e-5)
    
    # optimizer = AdamW(model.parameters(), lr=7e-5)

    criterion = torch.nn.CrossEntropyLoss()

    scheduler = LinearLR(
        optimizer,
        start_factor=0.5,
        total_iters=5
    )

    best_val_loss = float('inf')
    max_epochs = 10
    
    patience = 3
    no_improve_count = 0

    for epoch in range(max_epochs):
        model.train()
        total_train_loss = 0
        pbar = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1} [Train]"
        )

        for batch in pbar:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids)

            pad_id = tokenizer.pad_token_id
            seq_lengths = (input_ids != pad_id).sum(dim=1) - 1
            logits = outputs.logits[torch.arange(input_ids.size(0)), seq_lengths]
            
            loss = criterion(logits, labels)
            loss.backward()
            
            optimizer.step()
            total_train_loss += loss.item()

            pbar.set_postfix({
                'loss': f"{loss.item():.4f}"
            })

        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()

        total_val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():

            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                labels = batch['label'].to(device)

                outputs = model(input_ids)

                pad_id = tokenizer.pad_token_id
                seq_lengths = (input_ids != pad_id).sum(dim=1) - 1
                logits = outputs.logits[torch.arange(input_ids.size(0)), seq_lengths]
                
                loss = criterion(logits, labels)
                total_val_loss += loss.item()

                preds = torch.argmax(logits, dim=-1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_val_loss = total_val_loss / len(val_loader)
        val_accuracy = correct / total

        print(
            f"Epoch {epoch + 1} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Val Acc: {val_accuracy * 100:.2f}%"
        )

        scheduler.step()

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(
                model.state_dict(),
                'mamba_sst_2_1_4B_weights.pt'
            )
            print("Saved best model.")
        else:
            no_improve_count += 1
            print(f"No improvement ({no_improve_count}/{patience})")
            if no_improve_count >= patience:
                print("Early stopping.")
                break

    print("\nEvaluating on Test Set...")
    model.load_state_dict(
        torch.load('mamba_sst_2_1_4B_weights.pt')
    )

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids)
            pad_id = tokenizer.pad_token_id
            seq_lengths = (input_ids != pad_id).sum(dim=1) - 1
            logits = outputs.logits[torch.arange(input_ids.size(0)), seq_lengths]
            
            preds = torch.argmax(logits, dim=-1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_accuracy = correct / total
    print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")


device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

model.to(device)

train_sst_local(model, tokenizer, device)

Train size: 67349
Validation size: 872
Test size: 1821


Epoch 1 [Train]: 100%|██████████| 2105/2105 [28:24<00:00,  1.23it/s, loss=0.5242]


Epoch 1 | Train Loss: 0.5711 | Val Loss: 0.5451 | Val Acc: 82.00%
Saved best model.


Epoch 2 [Train]: 100%|██████████| 2105/2105 [28:25<00:00,  1.23it/s, loss=0.4403]


Epoch 2 | Train Loss: 0.4548 | Val Loss: 0.4675 | Val Acc: 84.06%
Saved best model.


...

Epoch 9 [Train]: 100%|██████████| 2105/2105 [35:07<00:00,  1.00s/it, loss=0.1179]


Epoch 9 | Train Loss: 0.3127 | Val Loss: 0.3024 | Val Acc: 89.79%
Saved best model.


Epoch 10 [Train]: 100%|██████████| 2105/2105 [49:22<00:00,  1.41s/it, loss=0.2456]


Epoch 10 | Train Loss: 0.3074 | Val Loss: 0.2946 | Val Acc: 89.79%
Saved best model.

Evaluating on Test Set...


OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 22.09 GiB of which 31.81 MiB is free. Process 537228 has 7.65 GiB memory in use. Process 597032 has 13.85 GiB memory in use. Process 610684 has 494.00 MiB memory in use. Of the allocated memory 7.30 GiB is allocated by PyTorch, and 71.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# !pip install gdown

# Import gdown
import gdown

# Define the file ID and the destination file name
file_id = '1RnIygUDodGeKPqbcEQTOYR5dztpF6X1b'  #SST-2
destination = 'mamba_sst2_weights.pt'

# Construct the URL
url = f'https://drive.google.com/uc?id={file_id}'

# Download the file
gdown.download(url, destination, quiet=False)

In [7]:
model.load_state_dict(
    torch.load('mamba_sst_2_1_4B_weights.pt', map_location=torch.device('cpu')),
    strict=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

# Make model explainable.
modified_model = ModifiedMambaForCausalLM(model, is_fast_forward_available=False)
modified_model.eval()
model.backbone.embeddings.requires_grad = False
pretrained_embeddings = model.backbone.embeddings

## Load dataset

Load SST-2 dataset.

In [8]:
validation_dataset = get_sst_dataset(
    tokenizer=tokenizer,
    truncation=False,
    max_length=None
    )

## Generate explanation

Generate explanation for one sample.

In [10]:
i = 413
input_ids = validation_dataset.__getitem__(i)['input_ids'].unsqueeze(0).to(device)
label = torch.tensor(validation_dataset.__getitem__(i)['label']).long().to(device)
idx = torch.where(input_ids == 0)[1] + 1
pad_positions = (input_ids[0] == 0).nonzero(as_tuple=True)[0]

if len(pad_positions) > 0:
    input_ids = input_ids[:, :pad_positions[0]]  # trim at first pad token
# if no padding, input_ids is already clean — no slicing needed
embeddings = pretrained_embeddings(input_ids)

R, prediction = relevance_propagation(
    model=modified_model,
    embeddings=embeddings,
    targets=label,
    n_classes=2
    )

## Visualization

For simplicity, we use the visualization utilities in Captum to display the results.

In [11]:
from captum.attr import visualization as viz

In [12]:
tokens = []
for id in input_ids[0][1: -2]:
    tokens.append(tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens([id.item()])))
attributions = R[0][1: -2]
attributions = attributions / attributions.max()

In [13]:
# Visualize the attributions
viz.visualize_text([viz.VisualizationDataRecord(
    attributions,
    torch.max(model(input_ids).logits[:, -1, :], dim=1).values.item(),
    torch.argmax(model(input_ids).logits[:, -1, :], dim=1).item(),
    true_class=label.item(),
    attr_class=label.item(),
    attr_score=attributions.sum(),
    raw_input_ids=tokens,
    convergence_score=None
)])

True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
0,0 (0.42),0,3.27,| start of text |> at least one scene is so disgusting that viewers may be hard pressed to retain their lunch


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
0,0 (0.42),0,3.27,| start of text |> at least one scene is so disgusting that viewers may be hard pressed to retain their lunch


In [14]:
if isinstance(attributions, np.ndarray):
    attributions_tensor = torch.from_numpy(attributions)
    print("here")
else:
    attributions_tensor = attributions
    

sorted_indices = torch.argsort(attributions_tensor, descending=True)

here


In [15]:
def get_morf_curve_logits(model, input_ids, sorted_indices, target_label):
    scores = []
    temp_input_ids = input_ids.clone()
    
    # 0. Initial Logit
    with torch.no_grad():
        logits = model(temp_input_ids) 
        # Select the target class logit for the last token
        initial_logit = logits[0, -1, target_label].item()
        scores.append(initial_logit)

    # 1. Iteratively mask tokens and record the logit
    for idx in sorted_indices:
        # Map back to the correct position in input_ids
        temp_input_ids[0, idx + 1] = tokenizer.pad_token_id 
        
        with torch.no_grad():
            logits = model(temp_input_ids)
            logit = logits[0, -1, target_label].item()
            scores.append(logit)
            
    return scores
morf_values = get_morf_curve_logits(modified_model, input_ids, sorted_indices, label.item())

In [16]:
def get_morf_curve_batched(model, embeddings, sorted_indices, target_label, chunk_size=8):
    seq_len = len(sorted_indices)
    base_embeddings = embeddings.clone().repeat(seq_len + 1, 1, 1)
    
    for i, idx in enumerate(sorted_indices):
        base_embeddings[i+1:, idx, :] = 0.0   # no +1, indices already correct
    
    all_logits = []
    with torch.no_grad():
        for i in range(0, base_embeddings.size(0), chunk_size):
            chunk = base_embeddings[i:i+chunk_size].to(device)
            logits = model(inputs_embeds=chunk)
            all_logits.extend(logits[:, -1, target_label].cpu().tolist())
    return all_logits

In [17]:
import numpy as np
from tqdm import tqdm

num_samples = min(1000, len(validation_dataset))
all_delta_afs = []

print(f"Evaluating {num_samples} samples from SST-2...")

for i in tqdm(range(num_samples)):
    sample = validation_dataset.__getitem__(i)
    input_ids = sample['input_ids'].unsqueeze(0).to(device)
    label = torch.tensor(sample['label']).long().to(device)
    
    pad_id = tokenizer.pad_token_id
    pad_positions = (input_ids[0] == pad_id).nonzero(as_tuple=True)[0]

    if len(pad_positions) > 0:
        idx = pad_positions[0]   # first padding token
        input_ids = input_ids[:, :idx]
    
    embeddings = pretrained_embeddings(input_ids)
    # Get pred_label PER SAMPLE
    with torch.no_grad():
        outputs = modified_model(inputs_embeds=embeddings)
        pred_label = torch.argmax(outputs[0, -1, :]).item()
    
    
    R, _ = relevance_propagation(
        model=modified_model, 
        embeddings=embeddings,
        targets=torch.tensor(pred_label).long().to(device), 
        n_classes=2
    )
    
    attr = R[0]
    if isinstance(attr, np.ndarray):
        attr = torch.from_numpy(attr)
    
    idx_morf = torch.argsort(attr[1:].abs(), descending=True) + 1
    morf_curve = get_morf_curve_batched(modified_model, embeddings, idx_morf, pred_label)
    # af_morf = np.trapz(morf_curve) / (len(morf_curve) - 1)
    
    idx_lerf = torch.argsort(attr[1:].abs(), descending=False) + 1
    lerf_curve = get_morf_curve_batched(modified_model, embeddings, idx_lerf, pred_label)
    # af_lerf = np.trapz(lerf_curve) / (len(lerf_curve) - 1)
    
    x = np.linspace(0, 1, len(morf_curve))
    af_morf = np.trapz(morf_curve, x)   # AUC over [0,1]
    af_lerf = np.trapz(lerf_curve, x)
    # delta_af = af_morf - af_lerf
    delta_af = af_lerf - af_morf
    all_delta_afs.append(delta_af)
    # del R, embeddings, input_ids, attr, morf_curve, lerf_curve
    # gc.collect()
    # if i % 10 == 0:
    #     torch.cuda.empty_cache()

# Final Result
mean_delta_af = np.mean(all_delta_afs)
print(f"\nMean Delta AF over {num_samples} samples: {mean_delta_af:.4f}")
print(f"Paper Reference (Mamba-130M SST-2): 1.978")

Evaluating 872 samples from SST-2...


100%|██████████| 872/872 [34:37<00:00,  2.38s/it]


Mean Delta AF over 872 samples: 0.9768
Paper Reference (Mamba-130M SST-2): 1.978
